# Function Attributes and Bound Methods
## Advanced Tutorial-Style Problems with Detailed Solutions

This is a new advanced problem set for the same topic, but it is intentionally written as a **guided tutorial**.

Instead of jumping directly from a problem statement to a finished answer, each section follows a progression:

1. inspect a small behavior,
2. make a prediction,
3. run an experiment,
4. explain the result,
5. solve a larger task in logical steps,
6. verify the mental model with assertions.

The emphasis is on understanding how a function stored on a class becomes a method when accessed through an instance.


## How to use this notebook

For every problem:

- read the non-code cells first,
- make a prediction before running the experiment,
- run one cell at a time,
- compare the result with your prediction,
- then study the solution steps.

Several examples intentionally trigger exceptions, but they catch and print them so the notebook can continue running.


# Problem 1 — Why does a zero-argument class function fail through an instance?


Start with a function written inside a class that declares no parameters.

Through the class, it behaves like a normal function. Through an instance, Python creates a bound method.

Your task is to explain why those two access paths behave differently.


In [1]:
class P1_Device:
    def ping():
        return "pong"

print(P1_Device.ping)
print(type(P1_Device.ping))
print(P1_Device.ping())


<function P1_Device.ping at 0x000001E4703D4EA0>
<class 'function'>
pong


## Think before running the next experiment

Create an instance mentally.

Before running the experiment, predict:

- the type of `device.ping`,
- whether `device.ping()` succeeds,
- which object Python will try to supply automatically.


In [2]:
device = P1_Device()

print(device.ping)
print(type(device.ping))

try:
    print(device.ping())
except TypeError as ex:
    print(type(ex).__name__, ex)


<bound method P1_Device.ping of <__main__.P1_Device object at 0x000001E460344440>>
<class 'method'>
TypeError P1_Device.ping() takes 0 positional arguments but 1 was given


## What did we observe?

The instance lookup returned a **method**, not the original plain function.

The failing call is evidence that the method supplied one positional argument even though the underlying function declares zero parameters.


### Solution step 1 — inspect the bound object

A bound method stores the object it belongs to in `__self__`.


### Solution step 2 — inspect the underlying function

The class function used to create the method is available through `__func__`.


### Solution step 3 — reconstruct the call

Conceptually, `device.ping()` behaves like `P1_Device.ping(device)`.


In [3]:
bound = device.ping

print(bound.__self__ is device)
print(bound.__func__ is P1_Device.ping)

try:
    P1_Device.ping(device)
except TypeError as ex:
    print(type(ex).__name__, ex)


True
True
TypeError P1_Device.ping() takes 0 positional arguments but 1 was given


## Takeaway

The important lesson is not the error itself. The important lesson is that **instance access changes the callable interface by binding an object as the first argument**.


# Problem 2 — Repair the method and prove that binding is object-specific


Now create a correctly shaped instance method.

The method should report a label stored on the instance.

Use two objects so we can prove that one underlying function can be bound to two different instances.


In [4]:
class P2_Device:
    def __init__(self, label):
        self.label = label

    def ping(self):
        return f"{self.label}: pong"

left_device = P2_Device("left")
right_device = P2_Device("right")


## Think before running the next experiment

Before running the next cell, decide which statements should be true:

- `left_device.ping.__func__ is right_device.ping.__func__`
- `left_device.ping.__self__ is left_device`
- `right_device.ping.__self__ is right_device`


In [5]:
print(left_device.ping())
print(right_device.ping())

print(left_device.ping.__func__ is right_device.ping.__func__)
print(left_device.ping.__self__ is left_device)
print(right_device.ping.__self__ is right_device)


left: pong
right: pong
True
True
True


## What did we observe?

Both method objects wrap the same function, but each one remembers a different instance.

That separation—same function, different `__self__`—is central to method binding.


### Solution step 1 — identify shared behavior

The function comes from the class, so all ordinary instances use the same function object unless it is overridden.


### Solution step 2 — identify per-object state

The bound method records the specific instance in `__self__`.


### Solution step 3 — verify explicitly

Use identity checks rather than relying only on printed representations.


In [6]:
assert left_device.ping.__func__ is P2_Device.ping
assert right_device.ping.__func__ is P2_Device.ping
assert left_device.ping.__self__ is left_device
assert right_device.ping.__self__ is right_device


## Takeaway

A useful mental model is: **bound method = function + instance**.


# Problem 3 — Is the name `self` special syntax?


Python code conventionally names the first instance-method parameter `self`.

But conventions and language rules are not the same thing.

Test whether changing that parameter name changes binding behavior.


In [7]:
class P3_Counter:
    def __init__(obj, start):
        obj.value = start

    def increment(current_object, amount=1):
        current_object.value += amount
        return current_object.value

counter = P3_Counter(10)


## Think before running the next experiment

Predict whether `counter.increment(5)` works.

If it works, what does that tell us about the spelling `self`?


In [8]:
print(counter.increment(5))
print(counter.value)
print(counter.increment.__self__ is counter)


15
15
True


## What did we observe?

The method works normally.

Python binding does not depend on the word `self`. It depends on the position of the first parameter and descriptor-based method binding.


### Solution step 1 — distinguish convention from mechanism

`self` is the community convention for the first instance parameter.


### Solution step 2 — identify what Python actually supplies

Python supplies the instance as the first positional argument when the bound method is called.


In [9]:
assert counter.value == 15
assert counter.increment.__func__ is P3_Counter.increment
assert counter.increment.__self__ is counter


## Takeaway

Use `self` in real code for clarity, even though Python does not require that exact name.


# Problem 4 — Compare `obj.method(x)` with `Class.method(obj, x)`


A bound method automatically supplies the instance.

Calling the function through the class lets us supply that first argument explicitly.

Use a method that mutates state so the equivalence is visible.


In [10]:
class P4_Wallet:
    def __init__(self, balance):
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount
        return self.balance

wallet = P4_Wallet(100)


## Think before running the next experiment

Predict the final balance after these two operations:

```python
wallet.deposit(25)
P4_Wallet.deposit(wallet, 50)
```


In [11]:
print(wallet.deposit(25))
print(P4_Wallet.deposit(wallet, 50))
print(wallet.balance)


125
175
175


## What did we observe?

Both forms called the same underlying function with the same wallet object as the first argument.

The difference was only whether the instance was supplied automatically or explicitly.


### Solution step 1 — inspect the method

Confirm that `wallet.deposit.__func__` is the class function.


### Solution step 2 — inspect the bound instance

Confirm that the method's `__self__` is the wallet.


In [12]:
assert wallet.deposit.__func__ is P4_Wallet.deposit
assert wallet.deposit.__self__ is wallet
assert wallet.balance == 175


## Takeaway

For ordinary instance methods, `obj.method(args...)` is behaviorally similar to `Class.method(obj, args...)`.


# Problem 5 — What exactly is stored in the class dictionary?


Attribute access performs work for us.

Inspecting `Class.__dict__` directly helps separate **storage** from **lookup behavior**.


In [13]:
class P5_Renderer:
    def render(self, value):
        return f"[{value}]"


## Think before running the next experiment

Predict the types of:

- `P5_Renderer.__dict__["render"]`
- `P5_Renderer.render`
- `P5_Renderer().render`


In [14]:
renderer = P5_Renderer()

print(P5_Renderer.__dict__["render"])
print(type(P5_Renderer.__dict__["render"]))
print(type(P5_Renderer.render))
print(type(renderer.render))


<function P5_Renderer.render at 0x000001E4703BC400>
<class 'function'>
<class 'function'>
<class 'method'>


## What did we observe?

The class dictionary contains the function.

The bound method appears when the function is retrieved through an instance.


### Solution step 1 — inspect the class namespace

The function exists under the method name in the class dictionary.


### Solution step 2 — inspect the instance namespace

The bound method does not need to be stored in `instance.__dict__`.


In [15]:
print("render in class dict:", "render" in P5_Renderer.__dict__)
print("render in instance dict:", "render" in renderer.__dict__)

assert "render" in P5_Renderer.__dict__
assert "render" not in renderer.__dict__


render in class dict: True
render in instance dict: False


## Takeaway

Method binding is usually created during lookup rather than by permanently storing a method object on each instance.


# Problem 6 — Are repeated method lookups identical objects?


Repeatedly evaluating `obj.method` may create fresh bound-method objects.

That raises a subtle question: what does it mean for two method lookups to represent the 'same method'?


In [16]:
class P6_Demo:
    def work(self):
        return "done"

demo6 = P6_Demo()
first_lookup = demo6.work
second_lookup = demo6.work


## Think before running the next experiment

Predict these results:

```python
first_lookup is second_lookup
first_lookup.__func__ is second_lookup.__func__
first_lookup.__self__ is second_lookup.__self__
```


In [17]:
print(first_lookup is second_lookup)
print(first_lookup.__func__ is second_lookup.__func__)
print(first_lookup.__self__ is second_lookup.__self__)


False
True
True


## What did we observe?

Object identity and logical binding are different ideas.

Two bound-method objects can be separate objects while still wrapping the same function and the same instance.


### Solution step 1 — avoid using only `is` on repeated lookup

The bound-method wrapper itself may be recreated.


### Solution step 2 — compare the components

Use `__func__` and `__self__` when you need to reason about binding structure.


In [18]:
assert first_lookup.__func__ is second_lookup.__func__
assert first_lookup.__self__ is second_lookup.__self__


## Takeaway

When debugging method binding, inspect the pieces instead of trusting only wrapper-object identity.


# Problem 7 — Store a bound method, then reassign the variable that named the object


A bound method is a first-class object.

Once you store it, it should continue to know which instance it was originally bound to.


In [19]:
class P7_Probe:
    def __init__(self, name):
        self.name = name

    def read(self):
        return self.name

probe_a = P7_Probe("A")
probe_b = P7_Probe("B")
saved_reader = probe_a.read


## Think before running the next experiment

Now imagine we execute:

```python
probe_a = probe_b
```

Should `saved_reader()` start reading from B, or stay bound to the original A object?


In [20]:
probe_a = probe_b

print("probe_a.read():", probe_a.read())
print("saved_reader():", saved_reader())


probe_a.read(): B
saved_reader(): A


## What did we observe?

Reassigning a variable does not rewrite an existing bound method.

The method already stores its target object.


### Solution step 1 — inspect the saved method

Look directly at `saved_reader.__self__`.


### Solution step 2 — separate variables from objects

A Python variable name can be rebound, but the method still references the original object.


In [21]:
print(saved_reader.__self__.name)
assert saved_reader.__self__.name == "A"
assert saved_reader.__func__ is P7_Probe.read


A


## Takeaway

Method binding belongs to the method object, not to the variable name you originally used to reach the instance.


# Problem 8 — Use bound methods as callbacks


Bound methods are practical because they package both the function and the target instance into one callable value.

Build a callback list from multiple instances.


In [22]:
class P8_Listener:
    def __init__(self, name):
        self.name = name

    def receive(self, message):
        return f"{self.name} received {message}"

listeners8 = [
    P8_Listener("A"),
    P8_Listener("B"),
    P8_Listener("C"),
]
callbacks8 = [listener.receive for listener in listeners8]


## Think before running the next experiment

If the callback list stores only callable objects, where is the target instance information kept?


In [23]:
for callback in callbacks8:
    print(callback("event"))


A received event
B received event
C received event


## What did we observe?

Each callback remembers a different instance even though the callback list itself does not store separate `(object, function)` pairs.


### Solution step 1 — inspect each callback's `__self__`


### Solution step 2 — compare their `__func__`

All callbacks should use the same class function.


In [24]:
for callback in callbacks8:
    print(
        callback.__self__.name,
        callback.__func__.__name__,
    )

assert len({id(cb.__func__) for cb in callbacks8}) == 1


A receive
B receive
C receive


## Takeaway

A bound method is often an ideal callback representation because the target object is already embedded in the callable.


# Problem 9 — Contrast bound callbacks with a late-binding closure bug


A loop that builds lambdas can accidentally capture a loop variable instead of a per-iteration value.

Compare that behavior with storing bound methods directly.


In [25]:
late_callbacks9 = []

for listener in listeners8:
    late_callbacks9.append(
        lambda: listener.receive("late")
    )


## Think before running the next experiment

Predict whether the lambdas call A, B, and C separately, or whether they all use the same final listener.


In [26]:
print([callback() for callback in late_callbacks9])

stable_callbacks9 = [
    listener.receive
    for listener in listeners8
]

print([
    callback("stable")
    for callback in stable_callbacks9
])


['C received late', 'C received late', 'C received late']
['A received stable', 'B received stable', 'C received stable']


## What did we observe?

The lambda version closes over the loop variable. The bound-method version captures the binding immediately when `listener.receive` is evaluated.


### Solution step 1 — identify what the lambda closes over

It closes over the variable `listener`.


### Solution step 2 — identify what a bound method stores

It stores a specific instance in `__self__`.


In [27]:
assert [cb.__self__.name for cb in stable_callbacks9] == ["A", "B", "C"]


## Takeaway

This is a practical case where understanding method binding leads to simpler and safer callback code.


# Problem 10 — Monkey-patch a class after an instance already exists


Functions assigned to a class at runtime can still participate in normal method binding.

Create the object first, then add the function to its class.


In [28]:
class P10_Record:
    def __init__(self, value):
        self.value = value

record10 = P10_Record(21)

def double10(self):
    return self.value * 2


## Think before running the next experiment

If we assign `P10_Record.double = double10`, should the already-created `record10` instance gain a bound method on its next lookup?


In [29]:
P10_Record.double = double10

print(record10.double)
print(record10.double())


<bound method double10 of <__main__.P10_Record object at 0x000001E47032F0E0>>
42


## What did we observe?

The instance did not need to be recreated.

The class now contains a function descriptor under `double`, so the next instance lookup binds it normally.


### Solution step 1 — inspect the class dictionary


### Solution step 2 — inspect the bound method pieces


In [30]:
assert P10_Record.__dict__["double"] is double10
assert record10.double.__func__ is double10
assert record10.double.__self__ is record10


## Takeaway

Runtime class modification and original class-body definitions participate in the same binding mechanism.


# Problem 11 — Assign a function directly to an instance


Now perform a superficially similar operation, but assign the function to the **instance** instead of the class.

This is where many learners expect automatic binding that does not occur.


In [31]:
def triple11(self):
    return self.value * 3

record10.triple = triple11


## Think before running the next experiment

Predict the type of `record10.triple`.

Will calling `record10.triple()` automatically pass `record10`?


In [32]:
print(record10.triple)
print(type(record10.triple))

try:
    print(record10.triple())
except TypeError as ex:
    print(type(ex).__name__, ex)


<function triple11 at 0x000001E4703D5620>
<class 'function'>
TypeError triple11() missing 1 required positional argument: 'self'


## What did we observe?

The function is stored directly in the instance dictionary, so it is returned as an ordinary function object.


### Solution step 1 — inspect the instance dictionary


### Solution step 2 — call the function explicitly

If it expects the object as its first argument, pass the instance yourself.


In [33]:
print(record10.__dict__)
print(record10.triple(record10))

assert "triple" in record10.__dict__
assert "triple" not in P10_Record.__dict__


{'value': 21, 'triple': <function triple11 at 0x000001E4703D5620>}
63


## Takeaway

Class-level function assignment enables normal binding; direct instance assignment does not automatically create a method.


# Problem 12 — Manually bind an instance-level function


Sometimes per-instance method behavior is intentional.

Use `types.MethodType` to turn a standalone function into a bound method for one specific object.


In [34]:
import types

def quadruple12(self):
    return self.value * 4


## Think before running the next experiment

What should `types.MethodType(quadruple12, record10)` contain in its `__func__` and `__self__` attributes?


In [35]:
record10.quadruple = types.MethodType(
    quadruple12,
    record10,
)

print(record10.quadruple)
print(type(record10.quadruple))
print(record10.quadruple())


<bound method quadruple12 of <__main__.P10_Record object at 0x000001E47032F0E0>>
<class 'method'>
84


## What did we observe?

This time the instance attribute itself is already a bound method object.


### Solution step 1 — verify the underlying function


### Solution step 2 — verify the target instance


In [36]:
assert record10.quadruple.__func__ is quadruple12
assert record10.quadruple.__self__ is record10
assert record10.quadruple() == 84


## Takeaway

`types.MethodType` is an explicit way to create the function-plus-instance pairing yourself.


# Problem 13 — Bind a function manually with `__get__`


Normal Python functions implement descriptor behavior.

Call that behavior directly to see how a function turns into a method.


In [37]:
class P13_Scaler:
    def __init__(self, factor):
        self.factor = factor

    def scale(self, value):
        return self.factor * value

scaler13 = P13_Scaler(5)
raw_scale13 = P13_Scaler.__dict__["scale"]


## Think before running the next experiment

Predict what `raw_scale13.__get__(scaler13, P13_Scaler)` returns.

Should it resemble `scaler13.scale`?


In [38]:
manual_scale13 = raw_scale13.__get__(
    scaler13,
    P13_Scaler,
)

automatic_scale13 = scaler13.scale

print(manual_scale13)
print(automatic_scale13)
print(manual_scale13(6))


<bound method P13_Scaler.scale of <__main__.P13_Scaler object at 0x000001E47032F230>>
<bound method P13_Scaler.scale of <__main__.P13_Scaler object at 0x000001E47032F230>>
30


## What did we observe?

The manual descriptor call and normal instance lookup create method bindings around the same function and instance.


### Solution step 1 — compare `__func__`


### Solution step 2 — compare `__self__`


In [39]:
assert manual_scale13.__func__ is automatic_scale13.__func__
assert manual_scale13.__self__ is automatic_scale13.__self__
assert manual_scale13(6) == automatic_scale13(6) == 30


## Takeaway

The function's `__get__` method exposes the mechanism behind ordinary instance-method binding.


# Problem 14 — What does function `__get__` do for class-style access?


The descriptor protocol also needs to handle access when there is no instance.

Call `__get__` with `instance=None`.


In [40]:
class_side14 = raw_scale13.__get__(
    None,
    P13_Scaler,
)


## Think before running the next experiment

Should class-style descriptor access create a bound method, or leave the function unbound?


In [41]:
print(class_side14)
print(type(class_side14))
print(class_side14 is raw_scale13)


<function P13_Scaler.scale at 0x000001E4703D5940>
<class 'function'>
True


## What did we observe?

Without an instance, the normal function descriptor returns the function itself rather than a method bound to an object.


### Solution step 1 — connect this with normal syntax

`Class.method` resembles this class-side descriptor access.


### Solution step 2 — contrast with instance syntax

`instance.method` provides an instance and therefore produces a bound method.


In [42]:
assert class_side14 is raw_scale13
assert type(class_side14).__name__ == "function"


## Takeaway

The presence or absence of an instance determines whether the function is bound.


# Problem 15 — Shadow a method with an instance attribute


An ordinary method function is a non-data descriptor.

An instance attribute with the same name can therefore shadow normal method lookup.


In [43]:
class P15_Job:
    def run(self):
        return "class implementation"

job15 = P15_Job()
print(job15.run())


class implementation


## Think before running the next experiment

What happens if the instance stores a string under the name `run`?

Will normal lookup still find the class method first?


In [44]:
job15.run = "disabled"

print(job15.run)

try:
    job15.run()
except TypeError as ex:
    print(type(ex).__name__, ex)


disabled
TypeError 'str' object is not callable


## What did we observe?

Normal instance lookup found the instance attribute, so the class function was not used.


### Solution step 1 — prove that the class function still exists


### Solution step 2 — remove the shadowing attribute

Deleting the instance attribute should restore normal method lookup.


In [45]:
print(P15_Job.run(job15))

del job15.run

print(job15.run())

assert job15.run() == "class implementation"


class implementation
class implementation


## Takeaway

An instance can accidentally disable or replace access to a normal class method simply by storing an attribute with the same name.


# Problem 16 — Replace a class function after saving a bound method


A saved bound method already contains both `__func__` and `__self__`.

Test whether replacing the class function changes an old method object retroactively.


In [46]:
class P16_Service:
    def execute(self):
        return "version 1"

service16 = P16_Service()
old_execute16 = service16.execute

def execute_v2_16(self):
    return "version 2"


## Think before running the next experiment

After assigning `P16_Service.execute = execute_v2_16`, predict:

- `old_execute16()`
- `service16.execute()`


In [47]:
P16_Service.execute = execute_v2_16

print("old:", old_execute16())
print("new lookup:", service16.execute())


old: version 1
new lookup: version 2


## What did we observe?

The old method kept the function it already stored. A new lookup sees the replacement function on the class.


### Solution step 1 — inspect the old method's `__func__`


### Solution step 2 — inspect the fresh lookup's `__func__`


In [48]:
print(old_execute16.__func__)
print(service16.execute.__func__)

assert old_execute16() == "version 1"
assert service16.execute() == "version 2"
assert old_execute16.__func__ is not service16.execute.__func__


<function P16_Service.execute at 0x000001E4703D60C0>
<function execute_v2_16 at 0x000001E4703D6020>


## Takeaway

Bound methods are objects with captured binding state, not live references that re-resolve the class function every time they are called.


# Problem 17 — Inherited methods bind to subclass instances


Method binding and inheritance work together.

A function may be defined on a base class but bound to an instance of a subclass.


In [49]:
class P17_Base:
    def identify(self):
        return type(self).__name__

class P17_Child(P17_Base):
    pass

child17 = P17_Child()
method17 = child17.identify


## Think before running the next experiment

Predict:

- where `method17.__func__` comes from,
- which object is stored in `method17.__self__`.


In [50]:
print(method17.__func__)
print(method17.__self__)
print(method17())


<function P17_Base.identify at 0x000001E4703D6340>
P17_Child


## What did we observe?

The function comes from the base class, but the binding targets the child instance.


### Solution step 1 — verify function ownership


### Solution step 2 — verify instance binding


In [51]:
assert method17.__func__ is P17_Base.identify
assert method17.__self__ is child17
assert method17() == "P17_Child"


## Takeaway

Inheritance chooses where the function is found; binding chooses which instance becomes the first argument.


# Problem 18 — Override a method and explicitly call the base implementation


Now the subclass defines its own function under the same method name.

Normal lookup should use the subclass function, but the base function still exists.


In [52]:
class P18_Base:
    def identify(self):
        return "base"

class P18_Child(P18_Base):
    def identify(self):
        return "child"

child18 = P18_Child()


## Think before running the next experiment

Predict the outputs of:

```python
child18.identify()
P18_Base.identify(child18)
```


In [53]:
print(child18.identify())
print(P18_Base.identify(child18))


child
base


## What did we observe?

Normal lookup selects the override. Explicit class-function access lets us choose the base implementation and supply the child instance ourselves.


### Solution step 1 — inspect the normally bound function


### Solution step 2 — manually bind the base function to the child


In [54]:
base_bound18 = P18_Base.identify.__get__(
    child18,
    P18_Child,
)

print(base_bound18())
assert child18.identify.__func__ is P18_Child.identify
assert base_bound18.__func__ is P18_Base.identify
assert base_bound18.__self__ is child18


base


## Takeaway

The function chosen and the object bound are separate parts of the method model.


# Problem 19 — Function metadata is still reachable through a bound method


Functions are objects and may carry custom attributes.

Attach metadata to a class function, then retrieve the method through an instance.


In [55]:
class P19_Endpoint:
    def handle(self, payload):
        return payload

P19_Endpoint.handle.route = "/items"
P19_Endpoint.handle.permission = "write"

endpoint19 = P19_Endpoint()
method19 = endpoint19.handle


## Think before running the next experiment

Where should the metadata live?

On the bound-method wrapper, the instance, or the underlying function?


In [56]:
print(method19.__func__.route)
print(method19.__func__.permission)


/items
write


## What did we observe?

The metadata belongs to the shared function object.

The bound method exposes that function as `__func__`.


### Solution step 1 — prove that the method points to the class function


### Solution step 2 — verify that separate instances share the same function metadata


In [57]:
other_endpoint19 = P19_Endpoint()

assert method19.__func__ is P19_Endpoint.handle
assert other_endpoint19.handle.__func__ is P19_Endpoint.handle
assert other_endpoint19.handle.__func__.route == "/items"


## Takeaway

Binding adds an instance reference; it does not duplicate the underlying function object for every instance.


# Problem 20 — Build a reusable method-inspection helper


At this point we repeatedly inspect `type`, `__func__`, and `__self__`.

Turn that pattern into a small debugging utility.


In [58]:
def inspect_callable20(obj):
    # TODO
    pass


## Think before running the next experiment

Not every callable is a bound method.

Design the helper so it does not crash when attributes such as `__func__` or `__self__` are missing.


In [59]:
def plain20(x):
    return x

class P20_Demo:
    def method(self, x):
        return x

demo20 = P20_Demo()
demo20.direct = lambda x: x


## What did we observe?

A robust inspector should use safe attribute access instead of assuming that every callable has method-specific attributes.


### Solution step 1 — use `getattr(..., default)`

That lets one helper inspect functions, methods, and instance-stored callables.


### Solution step 2 — test multiple callable shapes


In [60]:
def inspect_callable20(obj):
    return {
        "type": type(obj).__name__,
        "callable": callable(obj),
        "name": getattr(obj, "__name__", None),
        "func": getattr(obj, "__func__", None),
        "self": getattr(obj, "__self__", None),
    }

for candidate in [
    plain20,
    P20_Demo.method,
    demo20.method,
    demo20.direct,
]:
    print(inspect_callable20(candidate))


{'type': 'function', 'callable': True, 'name': 'plain20', 'func': None, 'self': None}
{'type': 'function', 'callable': True, 'name': 'method', 'func': None, 'self': None}
{'type': 'method', 'callable': True, 'name': 'method', 'func': <function P20_Demo.method at 0x000001E4703D6980>, 'self': <__main__.P20_Demo object at 0x000001E47032FA10>}
{'type': 'function', 'callable': True, 'name': '<lambda>', 'func': None, 'self': None}


## Takeaway

A good debugging habit is to ask: is this a function, a bound method, or merely some callable object?


# Problem 21 — Reconstruct a bound-method call manually


Write a helper that receives a bound method but is not allowed to call the method object directly.

Instead, rebuild the call from `__func__` and `__self__`.


In [61]:
def invoke_bound21(method, *args, **kwargs):
    # TODO
    pass


## Think before running the next experiment

The core expression should look like:

```python
method.__func__(method.__self__, ...)
```

But first decide how to reject objects that are not bound methods.


In [62]:
class P21_Calculator:
    def combine(self, a, b, *, factor=1):
        return (a + b) * factor

calculator21 = P21_Calculator()


## What did we observe?

This helper makes the implicit first argument completely explicit.


### Solution step 1 — validate expected attributes


### Solution step 2 — pass the stored instance first, followed by the user's arguments


In [63]:
def invoke_bound21(method, *args, **kwargs):
    if not hasattr(method, "__func__"):
        raise TypeError("expected a bound method")
    if not hasattr(method, "__self__"):
        raise TypeError("expected a bound method")

    return method.__func__(
        method.__self__,
        *args,
        **kwargs,
    )

result21 = invoke_bound21(
    calculator21.combine,
    2,
    3,
    factor=10,
)

print(result21)
assert result21 == 50


50


## Takeaway

Calling a bound method is conceptually simple once you can see its two stored pieces.


# Problem 22 — Compare two methods by logical binding


Repeated method lookup may create fresh wrapper objects.

Implement a helper that answers a more useful question: do these two method objects represent the same function bound to the same object?


In [64]:
def same_binding22(left, right):
    # TODO
    pass


## Think before running the next experiment

Which two identity comparisons should determine the answer?


In [65]:
class P22_Demo:
    def work(self):
        return "work"

a22 = P22_Demo()
b22 = P22_Demo()


## What did we observe?

The wrapper object's identity is not the same thing as binding identity.


### Solution step 1 — ensure both objects expose method metadata


### Solution step 2 — compare `__func__` and `__self__` with `is`


In [66]:
def same_binding22(left, right):
    return (
        hasattr(left, "__func__")
        and hasattr(left, "__self__")
        and hasattr(right, "__func__")
        and hasattr(right, "__self__")
        and left.__func__ is right.__func__
        and left.__self__ is right.__self__
    )

print(same_binding22(a22.work, a22.work))
print(same_binding22(a22.work, b22.work))

assert same_binding22(a22.work, a22.work)
assert not same_binding22(a22.work, b22.work)


True
False


## Takeaway

Logical method identity is often best expressed as the pair `(instance, function)`.


# Problem 23 — Build an event system that stores bound callbacks


Use bound methods in a small realistic design.

The event object should store callbacks and invoke them later without separately storing each listener object.


In [67]:
class P23_Event:
    def __init__(self):
        self._callbacks = []

    def subscribe(self, callback):
        # TODO
        pass

    def emit(self, *args, **kwargs):
        # TODO
        pass


## Think before running the next experiment

What does the event need to remember if the callback is already a bound method?

Hint: the target instance is already inside the callback.


In [68]:
class P23_Listener:
    def __init__(self, name):
        self.name = name

    def on_message(self, message):
        return f"{self.name}: {message}"

listener_a23 = P23_Listener("A")
listener_b23 = P23_Listener("B")


## What did we observe?

The event can store callable objects unchanged.

When those objects are bound methods, they already know where to send the call.


### Solution step 1 — reject non-callables


### Solution step 2 — append the callback


### Solution step 3 — emit by calling each stored callback


In [69]:
class P23_Event:
    def __init__(self):
        self._callbacks = []

    def subscribe(self, callback):
        if not callable(callback):
            raise TypeError("callback must be callable")
        self._callbacks.append(callback)

    def emit(self, *args, **kwargs):
        return [
            callback(*args, **kwargs)
            for callback in self._callbacks
        ]

event23 = P23_Event()
event23.subscribe(listener_a23.on_message)
event23.subscribe(listener_b23.on_message)

print(event23.emit("hello"))

for callback in event23._callbacks:
    print(callback.__self__.name, callback.__func__.__name__)


['A: hello', 'B: hello']
A on_message
B on_message


## Takeaway

This design works cleanly because bound methods are first-class callable objects with embedded target instances.


# Problem 24 — Unsubscribe a freshly retrieved bound method


Suppose an event stored `listener.on_message`.

Later, another evaluation of `listener.on_message` may be a fresh bound-method wrapper.

Use logical binding comparison so unsubscription does not depend on wrapper identity.


In [70]:
class P24_Event:
    def __init__(self):
        self._callbacks = []

    def subscribe(self, callback):
        self._callbacks.append(callback)

    def unsubscribe(self, callback):
        # TODO
        pass

    def emit(self, *args, **kwargs):
        return [
            cb(*args, **kwargs)
            for cb in self._callbacks
        ]


## Think before running the next experiment

How can the `same_binding22` helper be reused here?


In [71]:
event24 = P24_Event()


## What did we observe?

Unsubscription is a good example of why understanding the structure of a bound method can matter in application code.


### Solution step 1 — compare each stored callback with the callback being removed


### Solution step 2 — keep only the callbacks that do not represent the same binding


In [72]:
class P24_Event:
    def __init__(self):
        self._callbacks = []

    def subscribe(self, callback):
        self._callbacks.append(callback)

    def unsubscribe(self, callback):
        kept = []
        removed = False

        for existing in self._callbacks:
            if same_binding22(existing, callback):
                removed = True
            else:
                kept.append(existing)

        self._callbacks = kept
        return removed

    def emit(self, *args, **kwargs):
        return [
            cb(*args, **kwargs)
            for cb in self._callbacks
        ]

event24 = P24_Event()
event24.subscribe(listener_a23.on_message)
event24.subscribe(listener_b23.on_message)

print(event24.emit("before"))
print(event24.unsubscribe(listener_a23.on_message))
print(event24.emit("after"))


['A: before', 'B: before']
True
['B: after']


## Takeaway

The method wrapper may change between lookups, but the underlying `(function, instance)` pair can still identify the logical subscription.


# Problem 25 — Bound methods can affect object lifetime


A bound method stores a reference to its bound object.

That means a saved callback can keep an instance alive even after the original variable is deleted.


In [73]:
import weakref
import gc

class P25_Temporary:
    def ping(self):
        return "pong"

temporary25 = P25_Temporary()
weak25 = weakref.ref(temporary25)
saved25 = temporary25.ping


## Think before running the next experiment

After deleting `temporary25`, should `weak25()` immediately become `None` while `saved25` still exists?


In [74]:
del temporary25
gc.collect()

print("still alive:", weak25() is not None)
print("saved call:", saved25())


still alive: True
saved call: pong


## What did we observe?

The saved method's `__self__` keeps a strong reference to the instance.


### Solution step 1 — inspect the bound object through the saved method


### Solution step 2 — delete the method reference and collect again


In [75]:
print(saved25.__self__ is weak25())

del saved25
gc.collect()

print("collected:", weak25() is None)


True
collected: True


## Takeaway

Callback systems that store bound methods should be designed with object lifetime in mind.


# Problem 26 — Write a descriptor that mimics basic method binding


Now model the behavior ourselves.

The descriptor should return:

- the original function on class access,
- a bound method on instance access.


In [76]:
class P26_MethodLike:
    def __init__(self, function):
        self.function = function

    def __get__(self, instance, owner):
        # TODO
        pass


## Think before running the next experiment

What special value indicates class access rather than instance access?

How can `types.MethodType` create the bound object for us?


In [77]:
def p26_format(self, value):
    return f"{self.prefix}:{value}"


## What did we observe?

This custom descriptor is not replacing Python's full implementation. It is a teaching model of the central branch: class access versus instance access.


### Solution step 1 — when `instance is None`, return the function


### Solution step 2 — otherwise bind the function to the instance


In [78]:
class P26_MethodLike:
    def __init__(self, function):
        self.function = function

    def __get__(self, instance, owner):
        if instance is None:
            return self.function

        return types.MethodType(
            self.function,
            instance,
        )

class P26_Demo:
    def __init__(self, prefix):
        self.prefix = prefix

    format = P26_MethodLike(p26_format)

demo26 = P26_Demo("demo")

print(P26_Demo.format)
print(demo26.format)
print(demo26.format(10))

assert demo26.format.__self__ is demo26
assert demo26.format.__func__ is p26_format


<function p26_format at 0x000001E46EC56D40>
<bound method p26_format of <__main__.P26_Demo object at 0x000001E470438440>>
demo:10


## Takeaway

Descriptor logic explains why the same class attribute can appear as a function through the class and a method through an instance.


# Problem 27 — Build a tiny custom bound-method object


Go one level deeper.

Instead of using `types.MethodType`, create a small callable object that explicitly stores a function and an instance.


In [79]:
class P27_MiniBound:
    def __init__(self, function, instance):
        # TODO
        pass

    def __call__(self, *args, **kwargs):
        # TODO
        pass


## Think before running the next experiment

The call should ultimately perform:

```python
function(instance, *args, **kwargs)
```

What two attributes would make this object resemble a real bound method conceptually?


In [80]:
class P27_Target:
    def __init__(self, factor):
        self.factor = factor

    def multiply(self, value):
        return self.factor * value

target27 = P27_Target(7)


## What did we observe?

A miniature implementation helps separate the essential idea from Python's full internal machinery.


### Solution step 1 — store the function as `__func__`


### Solution step 2 — store the instance as `__self__`


### Solution step 3 — forward calls with the instance first


In [81]:
class P27_MiniBound:
    def __init__(self, function, instance):
        self.__func__ = function
        self.__self__ = instance

    def __call__(self, *args, **kwargs):
        return self.__func__(
            self.__self__,
            *args,
            **kwargs,
        )

mini27 = P27_MiniBound(
    P27_Target.multiply,
    target27,
)

print(mini27(6))
assert mini27(6) == 42
assert mini27.__func__ is P27_Target.multiply
assert mini27.__self__ is target27


42


## Takeaway

The conceptual essence of a bound method is simple: a callable wrapper that remembers both a function and an object.


# Problem 28 — Why can a decorator break an instance method?


A decorator replaces the function stored under a class attribute name.

Therefore the wrapper function itself becomes the function that Python binds to instances.


In [82]:
def bad_decorator28(function):
    def wrapper():
        return function()
    return wrapper

class P28_Broken:
    @bad_decorator28
    def action(self):
        return "ok"

broken28 = P28_Broken()


## Think before running the next experiment

The wrapper declares zero parameters.

What happens when instance access binds that wrapper and then the method is called?


In [83]:
print(P28_Broken.__dict__["action"])

try:
    broken28.action()
except TypeError as ex:
    print(type(ex).__name__, ex)


<function bad_decorator28.<locals>.wrapper at 0x000001E4703D7420>
TypeError bad_decorator28.<locals>.wrapper() takes 0 positional arguments but 1 was given


## What did we observe?

The wrapper—not the original function—is the class function that receives the automatically supplied instance.


### Solution step 1 — accept arbitrary positional and keyword arguments in the wrapper


### Solution step 2 — forward those arguments to the wrapped function


### Solution step 3 — preserve useful metadata with `functools.wraps`


In [84]:
from functools import wraps

def good_decorator28(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        return function(*args, **kwargs)
    return wrapper

class P28_Fixed:
    @good_decorator28
    def action(self, value):
        return value * 2

fixed28 = P28_Fixed()

print(fixed28.action(10))
assert fixed28.action(10) == 20
assert fixed28.action.__self__ is fixed28


20


## Takeaway

Method decorators must preserve the call shape needed for the bound instance and the method's remaining arguments.


# Problem 29 — Per-instance override without modifying the class


Sometimes you want one object to behave differently while leaving the class and all other instances unchanged.

Explicit binding makes that possible.


In [85]:
class P29_Speaker:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name}: default"

speaker_a29 = P29_Speaker("A")
speaker_b29 = P29_Speaker("B")

def custom_speak29(self):
    return f"{self.name}: custom"


## Think before running the next experiment

If `custom_speak29` is bound only to `speaker_a29`, what should happen to:

- `speaker_a29.speak()`
- `speaker_b29.speak()`
- `P29_Speaker.speak`


In [86]:
speaker_a29.speak = types.MethodType(
    custom_speak29,
    speaker_a29,
)

print(speaker_a29.speak())
print(speaker_b29.speak())


A: custom
B: default


## What did we observe?

Only one instance changed because the custom bound method was stored directly on that instance.


### Solution step 1 — verify the class function is unchanged


### Solution step 2 — inspect the custom instance binding


In [87]:
assert P29_Speaker.speak is speaker_b29.speak.__func__
assert speaker_a29.speak.__func__ is custom_speak29
assert speaker_a29.speak.__self__ is speaker_a29


## Takeaway

Per-instance binding is fundamentally different from monkey-patching the class, which affects normal lookup for all instances.


# Problem 30 — Rebind an existing method to another compatible instance


A bound method already gives us the underlying function.

Write a helper that takes that function and combines it with another object.


In [88]:
def rebind30(method, new_instance):
    # TODO
    pass

class P30_Label:
    def __init__(self, text):
        self.text = text

    def show(self):
        return self.text

label_a30 = P30_Label("A")
label_b30 = P30_Label("B")


## Think before running the next experiment

Which attribute of `label_a30.show` should be reused?

Which object should become the new `__self__`?


In [89]:
original30 = label_a30.show


## What did we observe?

Rebinding makes the separation between function and instance very explicit.


### Solution step 1 — validate that the input exposes `__func__`


### Solution step 2 — use `types.MethodType` with the new instance


In [90]:
def rebind30(method, new_instance):
    if not hasattr(method, "__func__"):
        raise TypeError("expected a bound method")

    return types.MethodType(
        method.__func__,
        new_instance,
    )

rebound30 = rebind30(
    original30,
    label_b30,
)

print(original30())
print(rebound30())

assert original30.__func__ is rebound30.__func__
assert original30.__self__ is label_a30
assert rebound30.__self__ is label_b30


A
B


## Takeaway

A method's behavior and target can be manipulated separately because they are stored separately.


# Problem 31 — Capstone: dynamic command registry


Build a small registry that stores bound methods as commands.

The registry should:

1. accept only bound instance methods,
2. execute a command by name,
3. describe the stored function and instance,
4. rebind a command to a different compatible instance.


In [91]:
class P31_CommandRegistry:
    def __init__(self):
        self._commands = {}

    def register(self, name, method):
        # TODO
        pass

    def execute(self, name, *args, **kwargs):
        # TODO
        pass

    def describe(self, name):
        # TODO
        pass

    def rebind(self, name, new_instance):
        # TODO
        pass


## Think before running the next experiment

Break the design into questions:

- How do we recognize a bound method?
- Why can execution call the stored object directly?
- Which attributes describe the binding?
- How can rebinding reuse the underlying function?


In [92]:
class P31_PricingService:
    def __init__(self, fee):
        self.fee = fee

    def quote(self, amount):
        return amount + self.fee

standard31 = P31_PricingService(5)
premium31 = P31_PricingService(50)


## What did we observe?

This capstone combines first-class method objects, `__func__`, `__self__`, and explicit rebinding.


### Solution step 1 — registration

Require `__func__` and a non-`None` `__self__`.


### Solution step 2 — execution

A stored bound method already knows its instance, so call it normally.


### Solution step 3 — description

Report the function's qualified name and the instance object.


### Solution step 4 — rebinding

Create a new method from the old `__func__` and the replacement instance.


In [93]:
class P31_CommandRegistry:
    def __init__(self):
        self._commands = {}

    def register(self, name, method):
        if not (
            hasattr(method, "__func__")
            and hasattr(method, "__self__")
            and method.__self__ is not None
        ):
            raise TypeError(
                "register expects a bound instance method"
            )

        self._commands[name] = method

    def execute(self, name, *args, **kwargs):
        return self._commands[name](
            *args,
            **kwargs,
        )

    def describe(self, name):
        method = self._commands[name]

        return {
            "command": name,
            "function": method.__func__.__qualname__,
            "instance_type": type(method.__self__).__name__,
            "instance": method.__self__,
        }

    def rebind(self, name, new_instance):
        old_method = self._commands[name]

        self._commands[name] = types.MethodType(
            old_method.__func__,
            new_instance,
        )

registry31 = P31_CommandRegistry()
registry31.register("quote", standard31.quote)

print(registry31.execute("quote", 100))
print(registry31.describe("quote"))

registry31.rebind("quote", premium31)

print(registry31.execute("quote", 100))
print(registry31.describe("quote"))

assert registry31.execute("quote", 100) == 150
assert registry31._commands["quote"].__func__ is P31_PricingService.quote
assert registry31._commands["quote"].__self__ is premium31


105
{'command': 'quote', 'function': 'P31_PricingService.quote', 'instance_type': 'P31_PricingService', 'instance': <__main__.P31_PricingService object at 0x000001E470438EC0>}
150
{'command': 'quote', 'function': 'P31_PricingService.quote', 'instance_type': 'P31_PricingService', 'instance': <__main__.P31_PricingService object at 0x000001E4703D0E10>}


## Takeaway

The command registry works because a bound method is a portable callable object that already combines implementation and target.


# Additional Worked Example — class assignment versus instance assignment

This final compact comparison puts two superficially similar operations side by side.

A function assigned to the class participates in normal function-descriptor binding.

The same function assigned directly to an instance is simply stored as an instance attribute unless you explicitly bind it.


In [94]:
class FinalTarget:
    pass

def final_external(self, value):
    return self, value

FinalTarget.class_side = final_external

final_target = FinalTarget()
final_target.instance_side = final_external

print("class_side:", final_target.class_side)
print("instance_side:", final_target.instance_side)

print("type(class_side):", type(final_target.class_side))
print("type(instance_side):", type(final_target.instance_side))

assert final_target.class_side.__self__ is final_target
assert final_target.class_side.__func__ is final_external
assert final_target.instance_side is final_external


class_side: <bound method final_external of <__main__.FinalTarget object at 0x000001E4704392B0>>
instance_side: <function final_external at 0x000001E47043C4A0>
type(class_side): <class 'method'>
type(instance_side): <class 'function'>


# Final Concept Review

Try answering these without executing new code.

1. What is stored in the class dictionary for a normal method definition?
2. What object is usually produced when that function is retrieved through an instance?
3. What does `method.__self__` contain?
4. What does `method.__func__` contain?
5. Why does an instance method normally need a first parameter such as `self`?
6. Is `self` a required keyword?
7. Why can `Class.method(instance, x)` often reproduce `instance.method(x)`?
8. Why does assigning a plain function directly to an instance not automatically bind it?
9. What does `types.MethodType(function, instance)` do?
10. What does `function.__get__(instance, owner)` demonstrate?
11. Why can repeated `instance.method` lookups represent the same binding without being identical objects?
12. How can an instance attribute shadow a normal method?
13. Does replacing a class function retroactively change bound methods that were already stored?
14. Can a base-class function bind to a subclass instance?
15. Why are bound methods convenient callbacks?
16. Why can storing a bound callback affect object lifetime?
17. What two identity checks are useful for logical method comparison?
18. Why can a badly written decorator break an instance method?
19. What is the simplest conceptual model of a bound method?
20. What is the difference between monkey-patching a class and explicitly binding behavior to one instance?


# Final Concept Review — Answers

1. A function object.
2. A bound method.
3. The specific object to which the method is bound.
4. The underlying function.
5. Instance method binding supplies the object as the first positional argument.
6. No. It is a convention.
7. The class-function form passes the instance explicitly instead of receiving it automatically from binding.
8. The function is retrieved directly from the instance dictionary rather than from the class as a function descriptor.
9. It explicitly creates a method bound to the supplied instance.
10. It exposes the function descriptor mechanism used for binding.
11. Each lookup may create a fresh wrapper while using the same `__func__` and `__self__`.
12. Store an instance attribute under the same name.
13. No. Existing bound method objects keep the function they already stored.
14. Yes.
15. They already package a function and a target instance into one callable object.
16. The bound method keeps a reference to its `__self__`.
17. Compare `left.__func__ is right.__func__` and `left.__self__ is right.__self__`.
18. The decorator's wrapper becomes the class function that receives the automatically supplied instance.
19. A function paired with a specific object.
20. Class modification changes normal lookup for instances generally; per-instance binding changes only that object's attribute.


# Closing Mental Model

When method behavior feels confusing, inspect it in this order:

1. **Where is the attribute stored?**
   - class dictionary?
   - instance dictionary?

2. **What object did lookup return?**
   - function?
   - bound method?
   - ordinary value?
   - another callable?

3. **If it is a method, inspect the two key pieces:**
   - `method.__func__`
   - `method.__self__`

4. **Translate the call mentally:**

```python
instance.method(a, b)
```

into the conceptual operation:

```python
underlying_function(instance, a, b)
```

5. **For runtime changes, distinguish assignment targets:**
   - assigning a function to the **class** enables normal function-descriptor binding,
   - assigning a function directly to the **instance** stores an ordinary instance attribute unless you explicitly bind it.

That model explains the progression in this notebook from simple class functions all the way to callbacks, monkey-patching, descriptors, decorators, and dynamic command registries.
